# Guía de Práctica N° 11
**Curso:** Análisis de Datos (SIS323) · Semestre Académico 2026-II  
**Unidad I:** Fundamentos de Análisis de Datos y Preprocesamiento  
**Institución:** Universidad Nacional del Altiplano – Puno (UNA-Puno)  
**Facultad:** Ingeniería Mecánica Eléctrica, Electrónica y Sistemas  
**Escuela Profesional:** Ingeniería de Sistemas  
**Docente:** Fredy Collanqui Martinez  
**Fecha:** 17/11/2026  

### Integrantes del equipo:
1. `MAMANI ZAPANA EDWARD` — Código: `229678`
2. `MAMANI PANCA ROGER GOLBER` — Código: `228917`
3. `GUTIERREZ ARRATIA KEVIN DIEGO` — Código: `227983`



## PASO 1 Variable objetivo y balance de clases

In [9]:
import pandas as pd
isp = pd.read_csv("clientesisp.csv")
print(isp.shape)
print(isp.drop(columns="cliente_id").head(3).T)
print()
print("Proporción de bajas:", round(isp["baja"].mean(), 3))

(900, 9)
                        0      1       2
zona               Urbana  Rural  Urbana
plan_mbps              10    100     100
meses_antiguedad       41     39      50
cortes_ultimo_mes       2      4       0
reclamos_6_meses        0      4       0
pago_puntual           No     Sí      Sí
precio_mensual         59    159     149
baja                    0      0       0

Proporción de bajas: 0.228


## PASO 2 Dividir con estratificación

In [10]:
from sklearn.model_selection import train_test_split
X = pd.get_dummies(isp.drop(columns=["cliente_id", "baja"]), drop_first=True, dtype=int)
y = isp["baja"]
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Predictoras:")
print(list(X.columns[:4]))
print(list(X.columns[4:]))
print(f"Bajas en entrenamiento: {y_ent.mean():.3f} | en prueba: {y_pru.mean():.3f}")

Predictoras:
['plan_mbps', 'meses_antiguedad', 'cortes_ultimo_mes', 'reclamos_6_meses']
['precio_mensual', 'zona_Urbana', 'pago_puntual_Sí']
Bajas en entrenamiento: 0.228 | en prueba: 0.227


## PASO 3 Escalar dentro de un pipeline

In [11]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
logistica = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_ent, y_ent)
coef = pd.Series(logistica[-1].coef_[0], index=X.columns).sort_values()
print("Coeficientes estandarizados:")
print(coef.round(2).to_string())


Coeficientes estandarizados:
meses_antiguedad    -0.80
pago_puntual_Sí     -0.27
precio_mensual      -0.20
zona_Urbana          0.01
plan_mbps            0.29
reclamos_6_meses     0.44
cortes_ultimo_mes    0.83


## PASO 4 Entrenar los tres modelos

In [12]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

arbol = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_ent, y_ent)
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(X_ent, y_ent)
arbol_libre = DecisionTreeClassifier(random_state=42).fit(X_ent, y_ent)

modelos = {
    "Regresión logística": logistica,
    "Árbol (profundidad 3)": arbol,
    "Árbol sin límite": arbol_libre,
    "KNN (k = 5)": knn
}

## PASO 5 Comparar exactitud en entrenamiento y en prueba

In [13]:
for nombre, m in modelos.items():
 ent, pru = m.score(X_ent, y_ent), m.score(X_pru, y_pru)
 print(f"{nombre:22s} exactitud: entrenamiento = {ent:.3f} prueba = {pru:.3f}")

Regresión logística    exactitud: entrenamiento = 0.815 prueba = 0.782
Árbol (profundidad 3)  exactitud: entrenamiento = 0.813 prueba = 0.760
Árbol sin límite       exactitud: entrenamiento = 0.997 prueba = 0.733
KNN (k = 5)            exactitud: entrenamiento = 0.828 prueba = 0.751
